In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import folium
from datetime import datetime


In [24]:
# Load dataset
# df = pd.read_csv("Credit card transactions - India - Simple.csv")
df = pd.read_csv(r"C:\Users\akind\Downloads\credit card (1)\Credit card transactions - India - Simple.csv")

In [25]:
# Display the shape of the dataset
print("Dataset Shape:", df.shape)
print("\nFirst 6 rows of the dataset:")
display(df.head(6))

Dataset Shape: (26052, 7)

First 6 rows of the dataset:


,index,City,Date,Card Type,Exp Type,Gender,Amount
0,0,"Delhi, India",29-Oct-14,Gold,Bills,F,82475
1,1,"Greater Mumbai, India",22-Aug-14,Platinum,Bills,F,32555
2,2,"Bengaluru, India",27-Aug-14,Silver,Bills,F,101738
3,3,"Greater Mumbai, India",12-Apr-14,Signature,Bills,F,123424
4,4,"Bengaluru, India",5-May-15,Gold,Bills,F,171574
5,5,"Delhi, India",8-Sep-14,Silver,Bills,F,100036


In [26]:
# Check for missing values
print("\nMissing values in each column:")
print(df.isnull().sum())

# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicate_count}")

# Verify data types
print("\nData types of columns:")
print(df.dtypes)


Missing values in each column:
index        0
City         0
Date         0
Card Type    0
Exp Type     0
Gender       0
Amount       0
dtype: int64

Number of duplicate rows: 0

Data types of columns:
index         int64
City         object
Date         object
Card Type    object
Exp Type     object
Gender       object
Amount        int64
dtype: object


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26052 entries, 0 to 26051
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   index      26052 non-null  int64 
 1   City       26052 non-null  object
 2   Date       26052 non-null  object
 3   Card Type  26052 non-null  object
 4   Exp Type   26052 non-null  object
 5   Gender     26052 non-null  object
 6   Amount     26052 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 1.4+ MB


In [28]:
# Summary statistics
print("\nSummary statistics of the dataset:")
print(df.describe(include='all'))
# Remove duplicates if any
df.drop_duplicates(inplace=True)


Summary statistics of the dataset:
               index              City       Date Card Type Exp Type Gender  \
count   26052.000000             26052      26052     26052    26052  26052   
unique           NaN               986        600         4        6      2   
top              NaN  Bengaluru, India  20-Sep-14    Silver     Food      F   
freq             NaN              3552         65      6840     5463  13680   
mean    13025.500000               NaN        NaN       NaN      NaN    NaN   
std      7520.708943               NaN        NaN       NaN      NaN    NaN   
min         0.000000               NaN        NaN       NaN      NaN    NaN   
25%      6512.750000               NaN        NaN       NaN      NaN    NaN   
50%     13025.500000               NaN        NaN       NaN      NaN    NaN   
75%     19538.250000               NaN        NaN       NaN      NaN    NaN   
max     26051.000000               NaN        NaN       NaN      NaN    NaN   

               

In [29]:
df = pd.read_csv(r"C:\Users\akind\Downloads\credit card (1)\Credit card transactions - India - Simple.csv")
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
df['City'] = df['City'].str.replace(', India','').str.strip()
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(int)

In [30]:
# Encode categorical features
for col in ['City','Card Type','Exp Type','Gender']:
    df[col+'_Code'] = LabelEncoder().fit_transform(df[col])

print(f"Loaded {len(df):,} transactions | Total Spend: ₹{df['Amount'].sum()/1e7:.1f} Crore")


Loaded 26,052 transactions | Total Spend: ₹407.5 Crore


In [31]:
# CREATE REALISTIC CUSTOMER IDs
df = df.sort_values(['City','Date'])
df['Days_Diff'] = df.groupby('City')['Date'].diff().dt.days.fillna(0)
df['New_Customer'] = (df['Days_Diff'] > 90) | (df['Days_Diff'] == 0)
df['Customer_ID'] = df['New_Customer'].cumsum()
df = df.drop(['Days_Diff','New_Customer'], axis=1)
print(f"Created {df['Customer_ID'].nunique():,} realistic customers")


Created 16,911 realistic customers


In [32]:
# EDA
print("\nTOP 10 CITIES BY SPEND")
top_cities = df.groupby('City')['Amount'].sum().sort_values(ascending=False).head(10)
px.bar(top_cities/1e7, title="Top 10 Cities by Spend (₹ Crore)", color_discrete_sequence=['#FF6B6B']).show()

px.sunburst(df.sample(5000), path=['Gender','Card Type','Exp Type'], values='Amount',
            title="Spending by Gender → Card → Category").show()



TOP 10 CITIES BY SPEND


In [34]:
import joblib


print("BANK-GRADE CREDIT CARD ANALYTICS SUITE - INDIA")
print("Loading your data...")

# ────────────────────────────── 1. LOAD & CLEAN DATA ──────────────────────────────
df = pd.read_csv("Credit card transactions - India - Simple.csv")
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
df['City'] = df['City'].str.replace(', India','').str.strip()
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(int)

# Encode categorical features
for col in ['City','Card Type','Exp Type','Gender']:
    df[col+'_Code'] = LabelEncoder().fit_transform(df[col])

print(f"Loaded {len(df):,} transactions | Total Spend: ₹{df['Amount'].sum()/1e7:.1f} Crore")

# ────────────────────────────── 2. CREATE REALISTIC CUSTOMER IDs ──────────────────────────────
df = df.sort_values(['City','Date'])
df['Days_Diff'] = df.groupby('City')['Date'].diff().dt.days.fillna(0)
df['New_Customer'] = (df['Days_Diff'] > 90) | (df['Days_Diff'] == 0)
df['Customer_ID'] = df['New_Customer'].cumsum()
df = df.drop(['Days_Diff','New_Customer'], axis=1)
print(f"Created {df['Customer_ID'].nunique():,} realistic customers")

# ────────────────────────────── 3. EDA (visualizations) ──────────────────────────────
print("\nTOP 10 CITIES BY SPEND")
top_cities = df.groupby('City')['Amount'].sum().sort_values(ascending=False).head(10)
px.bar(top_cities/1e7, title="Top 10 Cities by Spend (₹ Crore)", color_discrete_sequence=['#FF6B6B']).show()

px.sunburst(df.sample(5000), path=['Gender','Card Type','Exp Type'], values='Amount',
            title="Spending by Gender → Card → Category").show()

# ────────────────────────────── 4. FRAUD DETECTION (Autoencoder) ──────────────────────────────
features = ['Amount','Month','Day','Is_Weekend','City_Code','Card Type_Code','Exp Type_Code']
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Building and training autoencoder...")
input_layer = Input(shape=(7,))
encoded = Dense(32, activation='relu')(input_layer)
encoded = Dense(16, activation='relu')(encoded)
decoded = Dense(32, activation='relu')(encoded)
decoded = Dense(7, activation='linear')(decoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Training – start small to test
autoencoder.fit(
    X_scaled, X_scaled,
    epochs=12,           # good balance — try 10–15
    batch_size=1024,
    verbose=1,
    validation_split=0.1
)


print("Autoencoder training completed.")

# Continue with fraud scoring
recon = autoencoder.predict(X_scaled, verbose=0)
mse = np.mean(np.power(X_scaled - recon, 2), axis=1)
threshold = np.percentile(mse, 99.9)
df['AE_Fraud'] = (mse > threshold).astype(int)

print(f"\nFRAUD DETECTION: Autoencoder flagged {df['AE_Fraud'].sum()} suspicious transactions")

# ────────────────────────────── 5. INTERACTIVE INDIA FRAUD & SPENDING MAP ──────────────────────────────
city_coords = {
    'Bengaluru': [12.9716, 77.5946], 'Greater Mumbai': [19.0760, 72.8777], 'Delhi': [28.7041, 77.1025],
    'Ahmedabad': [23.0225, 72.5714], 'Hyderabad': [17.3850, 78.4867], 'Chennai': [13.0827, 80.2707],
    'Kolkata': [22.5726, 88.3639], 'Pune': [18.5204, 73.8567], 'Jaipur': [26.9124, 75.7873],
    'Lucknow': [26.8467, 80.9462], 'Kanpur': [26.4499, 80.3319], 'Surat': [21.1702, 72.8311]
}

map_data = df.groupby('City').agg({'Amount':'sum', 'AE_Fraud':'sum'}).reset_index()
map_data['Lat'] = map_data['City'].map(lambda x: city_coords.get(x, [20,78])[0])
map_data['Lon'] = map_data['City'].map(lambda x: city_coords.get(x, [20,78])[1])

m = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles="CartoDB positron")
for i, row in map_data.iterrows():
    color = 'red' if row['AE_Fraud'] > 0 else 'blue'
    folium.CircleMarker(
        [row['Lat'], row['Lon']], radius=np.log(row['Amount']+1)*3,
        popup=f"<b>{row['City']}</b><br>Spend: ₹{row['Amount']/1e7:.1f} Cr<br>Fraud Cases: {int(row['AE_Fraud'])}",
        color=color, fillOpacity=0.8
    ).add_to(m)
m.save("india_credit_map.html")
print("Interactive Map Saved → india_credit_map.html")

# ────────────────────────────── 6. CUSTOMER SEGMENTATION (RFM + K-Means) ──────────────────────────────
current_date = df['Date'].max()
rfm = df.groupby('Customer_ID').agg({
    'Date': lambda x: (current_date - x.max()).days,
    'Customer_ID': 'count',
    'Amount': 'sum'
}).rename(columns={'Date':'Recency','Customer_ID':'Frequency','Amount':'Monetary'})

rfm_scaled = StandardScaler().fit_transform(rfm)
rfm['Segment'] = KMeans(n_clusters=4, random_state=42).fit_predict(rfm_scaled)
rfm['Segment_Name'] = rfm['Segment'].map({0:'Champions',1:'Loyal',2:'At Risk',3:'Sleeping'})
print(f"\nCustomer Segments → {rfm['Segment_Name'].value_counts().to_dict()}")

# ────────────────────────────── 7. CHURN, CREDIT LIMIT & CLV ──────────────────────────────
customer_last = df.groupby('Customer_ID')['Date'].max().reset_index()
customer_last['Inactive_Days'] = (current_date - customer_last['Date']).dt.days
customer_last['Churned'] = (customer_last['Inactive_Days'] > 120).astype(int)

customer_spend = df.groupby('Customer_ID')['Amount'].mean().reset_index()
customer_spend['Recommended_Limit'] = (customer_spend['Amount'] * 3).round(-4)

monthly_spend = df.groupby('Customer_ID').apply(
    lambda x: x.set_index('Date').resample('M')['Amount'].sum().mean()
).reset_index(name='Avg_Monthly')
monthly_spend['CLV_Lakhs'] = (monthly_spend['Avg_Monthly'] * 12 * 5 / 100000).round(2)

# ────────────────────────────── 8. NEXT-MONTH HIGH-SPENDER PREDICTION ──────────────────────────────
print("\nPreparing next-month high-spender prediction...")

df_s = df.sort_values(['Customer_ID','Date']).copy()
df_s['Next30_Spend'] = df_s.groupby('Customer_ID')['Amount'].transform(
    lambda x: x.shift().rolling(30, min_periods=1).sum()
)

THRESHOLD = 200000
df_s['High_Spender_Next'] = (df_s['Next30_Spend'] > THRESHOLD).astype(int).fillna(0)

print(f"Positive high-spender cases (>{THRESHOLD}): {df_s['High_Spender_Next'].sum()} / {len(df_s):,}")

model = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

model.fit(df_s[features], df_s['High_Spender_Next'])

latest = df_s.drop_duplicates('Customer_ID', keep='last').copy()

proba = model.predict_proba(latest[features])


# Save the high-spender RandomForest model
joblib.dump(model, "models/high_spender_model.pkl")
print("High-spender model saved to: models/high_spender_model.pkl")

if proba.shape[1] == 2:
    latest['Big_Spender_Prob'] = proba[:, 1]
else:
    latest['Big_Spender_Prob'] = 0.0
    print("Warning: Only one class present in training data → probabilities set to 0")

# ────────────────────────────── 9. FINAL PREDICTIONS ──────────────────────────────
print("\n" + "="*80)
print("YOUR FINAL PREDICTIONS")
print("="*80)

print("1. TOP 10 NEXT-MONTH HIGH-SPENDERS")
print(f"(Probability of spending > ₹{THRESHOLD:,} in next 30 days)")
display(
    latest[['Customer_ID', 'City', 'Card Type', 'Big_Spender_Prob']]
    .sort_values('Big_Spender_Prob', ascending=False)
    .head(10)
    .style.format({'Big_Spender_Prob': '{:.1%}'})
)

print("\n2. TOP 10 MOST VALUABLE CUSTOMERS (CLV)")
display(
    monthly_spend.sort_values('CLV_Lakhs', ascending=False)
    .head(10)
    .style.format({'CLV_Lakhs': '₹{:.2f} Lakhs'})
)

print("\n3. TOP 10 FRAUD ALERTS")
display(
    df[df['AE_Fraud']==1]
    .sort_values('Amount', ascending=False)
    .head(10)[['Customer_ID','Date','City','Amount','Exp Type','Card Type']]
)

print("\n4. TOP 10 CUSTOMERS FOR CREDIT LIMIT INCREASE")
display(
    customer_spend.sort_values('Amount', ascending=False)
    .head(10)[['Customer_ID','Amount','Recommended_Limit']]
    .style.format({'Amount': '₹{:,.0f}', 'Recommended_Limit': '₹{:,.0f}'})
)

# ────────────────────────────── 10. FINAL DASHBOARD ──────────────────────────────
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Indian Credit Card Intelligence", layout="wide")
st.title("Bank-Grade Credit Card Analytics - India")

try:
    df = pd.read_csv("Credit card transactions - India - Simple.csv")
    df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
    df['City'] = df['City'].str.replace(', India','').str.strip()
except Exception as e:
    st.error(f"Error loading data: {e}")
    st.stop()

tab1, tab2, tab3 = st.tabs(["Overview", "Fraud Map", "Predictions"])

with tab1:
    c1, c2, c3 = st.columns(3)
    c1.metric("Total Spend", f"₹{df['Amount'].sum()/1e7:.1f} Cr")
    c2.metric("Transactions", f"{len(df):,}")
    c3.metric("Avg Transaction", f"₹{df['Amount'].mean():,.0f}")
    st.plotly_chart(px.line(df.groupby('Date')['Amount'].sum().reset_index(), x='Date', y='Amount'))

with tab2:
    try:
        with open("india_credit_map.html", "r", encoding="utf-8") as f:
            html_content = f.read()
        st.components.v1.html(html_content, height=600)
    except Exception as e:
        st.warning(f"Could not load map: {e}")

with tab3:
    st.write("High-value & at-risk customers ready for action")
'''

with open("final_dashboard.py", "w", encoding="utf-8") as f:
    f.write(dashboard_code)

print("\\nFINAL DASHBOARD → final_dashboard.py (Run: streamlit run final_dashboard.py)")
print("Map → india_credit_map.html")


BANK-GRADE CREDIT CARD ANALYTICS SUITE - INDIA
Loading your data...
Loaded 26,052 transactions | Total Spend: ₹407.5 Crore
Created 16,911 realistic customers

TOP 10 CITIES BY SPEND


Building and training autoencoder...
Epoch 1/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - loss: 0.8152 - val_loss: 1.5992
Epoch 2/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - loss: 0.6482 - val_loss: 1.3623
Epoch 3/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.4623 - val_loss: 1.0275
Epoch 4/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.2798 - val_loss: 0.6997
Epoch 5/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.1557 - val_loss: 0.3924
Epoch 6/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 0.0798 - val_loss: 0.2318
Epoch 7/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0468 - val_loss: 0.1722
Epoch 8/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0340 - val_loss: 0.1322
Epoch 9/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0263 - val_loss: 0.1079
Epoch 10/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0213 - val_loss: 0.0887
Epoch 11/12
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - loss: 0.0178 - val_loss: 0.0762
Epoch 12/12
23/23 ━━━━━━━━

,Customer_ID,City,Card Type,Big_Spender_Prob
21655,14430,Lucknow,Signature,90.0%
253,1202,Ahmedabad,Platinum,89.2%
21607,15273,Pune,Platinum,89.2%
16064,12833,Jagdalpur,Gold,89.0%
19753,14552,Mandsaur,Signature,88.8%
21197,15383,Pune,Gold,88.2%
23640,13148,Jaipur,Platinum,88.0%
291,5339,Bengaluru,Platinum,87.4%
23485,13050,Jaipur,Silver,87.0%
22636,12759,Hyderabad,Gold,87.0%



2. TOP 10 MOST VALUABLE CUSTOMERS (CLV)


,Customer_ID,Avg_Monthly,CLV_Lakhs
14429,14430,2376751.000000,₹1426.05 Lakhs
12770,12771,2012143.000000,₹1207.29 Lakhs
16468,16469,1952383.000000,₹1171.43 Lakhs
16221,16222,1821876.000000,₹1093.13 Lakhs
12521,12522,1764781.000000,₹1058.87 Lakhs
12987,12988,1577800.000000,₹946.68 Lakhs
16479,16480,1562165.000000,₹937.30 Lakhs
12644,12645,1544433.000000,₹926.66 Lakhs
13699,13700,1502593.000000,₹901.56 Lakhs
6198,6199,1485386.000000,₹891.23 Lakhs



3. TOP 10 FRAUD ALERTS


,Customer_ID,Date,City,Amount,Exp Type,Card Type
81,613,2014-02-02,Ahmedabad,934205,Bills,Silver
19216,16115,2014-10-16,Suar,298349,Bills,Silver
16594,16059,2015-03-09,Sirsi,296541,Bills,Silver
19057,16904,2014-09-17,Zira,294291,Bills,Gold
16084,16774,2015-01-23,Valsad,287089,Bills,Platinum
21427,16221,2014-01-04,Surat,285777,Bills,Silver
16067,16902,2013-12-30,Zira,283129,Bills,Gold
18553,16589,2015-02-23,Thiruvananthapuram,280203,Bills,Gold
18082,16769,2013-12-03,Valparai,261768,Bills,Silver
17400,16866,2014-03-25,Wardha,253759,Bills,Gold



4. TOP 10 CUSTOMERS FOR CREDIT LIMIT INCREASE


,Customer_ID,Amount,Recommended_Limit
6615,6616,"₹996,754","₹2,990,000"
7648,7649,"₹994,537","₹2,980,000"
9508,9509,"₹990,700","₹2,970,000"
623,624,"₹955,468","₹2,870,000"
1231,1232,"₹934,683","₹2,800,000"
7509,7510,"₹919,877","₹2,760,000"
1542,1543,"₹852,806","₹2,560,000"
4055,4056,"₹845,653","₹2,540,000"
10688,10689,"₹835,872","₹2,510,000"
5095,5096,"₹835,177","₹2,510,000"


\nFINAL DASHBOARD → final_dashboard.py (Run: streamlit run final_dashboard.py)
Map → india_credit_map.html


In [ ]:
from pathlib import Path
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print("Saving autoencoder and artifacts...")

# Save the trained autoencoder
autoencoder.save(MODEL_DIR / "autoencoder.keras")

# Save the scaler (you used it for X_scaled)
joblib.dump(scaler, MODEL_DIR / "scaler.pkl")

# Save label encoders (for API input encoding)
joblib.dump(LabelEncoder().fit(df['City']),      MODEL_DIR / "le_city.pkl")
joblib.dump(LabelEncoder().fit(df['Card Type']), MODEL_DIR / "le_card.pkl")
joblib.dump(LabelEncoder().fit(df['Exp Type']),  MODEL_DIR / "le_exp.pkl")

# Save the fraud threshold (for API comparison)
joblib.dump(threshold, MODEL_DIR / "fraud_threshold.pkl")

print(f"Saved {len(list(MODEL_DIR.glob('*')))} files to: {MODEL_DIR.resolve()}")

Saving autoencoder and artifacts...
Saved 7 files to: C:\Users\akind\OneDrive\Desktop\My_Project\Indian_Credit_card\models


In [35]:
# Save the fully processed DataFrame (with Customer_ID, AE_Fraud, etc.)
df.to_csv("processed_transactions.csv", index=False)
print("Processed data saved to: processed_transactions.csv")
print("Shape of saved data:", df.shape)
print("Columns in saved data:", df.columns.tolist())

Processed data saved to: processed_transactions.csv
Shape of saved data: (26052, 18)
Columns in saved data: ['index', 'City', 'Date', 'Card Type', 'Exp Type', 'Gender', 'Amount', 'Year', 'Month', 'Day', 'DayOfWeek', 'Is_Weekend', 'City_Code', 'Card Type_Code', 'Exp Type_Code', 'Gender_Code', 'Customer_ID', 'AE_Fraud']


In [36]:
import pandas as pd

file_path = "processed_transactions.csv"  # or the exact name you used

try:
    temp_df = pd.read_csv(file_path)
    print("File loaded successfully!")
    print("Shape:", temp_df.shape)
    print("\nColumns in the file:")
    print(temp_df.columns.tolist())
    print("\nDoes 'Customer_ID' exist?", 'Customer_ID' in temp_df.columns)
    print("Does 'AE_Fraud' exist?", 'AE_Fraud' in temp_df.columns)
    print("\nFirst 3 rows (preview):")
    print(temp_df.head(3))
except FileNotFoundError:
    print(f"File '{file_path}' not found in current directory.")
except Exception as e:
    print("Error loading file:", str(e))

File loaded successfully!
Shape: (26052, 18)

Columns in the file:
['index', 'City', 'Date', 'Card Type', 'Exp Type', 'Gender', 'Amount', 'Year', 'Month', 'Day', 'DayOfWeek', 'Is_Weekend', 'City_Code', 'Card Type_Code', 'Exp Type_Code', 'Gender_Code', 'Customer_ID', 'AE_Fraud']

Does 'Customer_ID' exist? True
Does 'AE_Fraud' exist? True

First 3 rows (preview):
   index      City        Date  Card Type Exp Type Gender  Amount  Year  \
0  19373  Achalpur  2013-12-06     Silver  Grocery      F  201032  2013   
1  17041  Achalpur  2014-02-06   Platinum  Grocery      F  178612  2014   
2  18871  Achalpur  2014-04-19  Signature     Fuel      M  138246  2014   

   Month  Day  DayOfWeek  Is_Weekend  City_Code  Card Type_Code  \
0     12    6          4           0          0               3   
1      2    6          3           0          0               1   
2      4   19          5           1          0               2   

   Exp Type_Code  Gender_Code  Customer_ID  AE_Fraud  
0          